In [1]:
pwd

'd:\\Kidney_classification_Using_MLOPS_and_DVC_Data-version-control\\research'

In [2]:
import os
from pathlib import Path

os.chdir("d:/Kidney_classification_Using_MLOPS_and_DVC_Data-version-control")
print("Working directory:", os.getcwd())

Working directory: d:\Kidney_classification_Using_MLOPS_and_DVC_Data-version-control


In [3]:
pwd

'd:\\Kidney_classification_Using_MLOPS_and_DVC_Data-version-control'

In [4]:
from dataclasses import dataclass
from pathlib import Path
@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [5]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

In [6]:
from cnnClassifier.config.configuration import ConfigurationManager

In [7]:
import zipfile
import gdown
from cnnClassifier import logger
from cnnClassifier.utils.common import get_size

In [8]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])
        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )
        return data_ingestion_config

In [9]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            gdown.download(self.config.source_URL, str(self.config.local_data_file), quiet=False, fuzzy=True)
            logger.info(f"Downloaded data to {self.config.local_data_file}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")

    def extract_zip_file(self):
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [10]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-03-01 15:45:30,742: INFO: common]: YAML file loaded successfully: config\config.yaml
[2026-03-01 15:45:30,744: INFO: common]: YAML file loaded successfully: params.yaml
[2026-03-01 15:45:30,745: INFO: common]: Created directory: artifacts
[2026-03-01 15:45:30,747: INFO: common]: Created directory: artifacts/data_ingestion
[2026-03-01 15:45:30,748: INFO: 1034350585]: File already exists of size: ~ 56208 KB
